# Project 02 (medium): the honest model race — pipelines, CV and tuning

**Goal:** compare several model families fairly via pipeline + cross-validation, tune the
best two, and measure honestly on the test set **once, right at the end**.

**Data:** Breast Cancer Wisconsin (Diagnostic), built into scikit-learn — 569 samples,
30 numerical features from images of cell nuclei, target: *malignant* (0) vs. *benign* (1).

Prior knowledge: script sections 1.4-1.5 and 2.1-2.5. Less guidance than project 01 —
you are to derive the construction of the pipelines and the grid from the script yourself.

## 1. Load, explore and split the data

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0 = malignant, 1 = benign

print(X.shape)
print(pd.Series(y).map({0: "malignant", 1: "benign"}).value_counts(normalize=True))

**Task:** make the split — stratified (script 1.4: keep the class shares equal in every
split), `test_size=0.2`, `random_state=42`. From now on the test set is **not** touched
until step 6.

In [ ]:
# TODO: X_train, X_test, y_train, y_test = train_test_split(...)

print(f"Training: {len(X_train)}, test: {len(X_test)}")
# Mini check: the class share in the test set is close to the overall share (stratified)
print(abs(y_test.mean() - y.mean()) < 0.02)

## 2. Baseline

Every serious model has to beat the `DummyClassifier` (always the majority class).
This is the zero line against which you can place all the CV results in a moment.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Dummy accuracy: {dummy.score(X_test, y_test):.3f}  (its ROC-AUC would be 0.5 - chance)")

## 3. Pipelines for six model families

**Task:** build a dictionary `models = {"name": Pipeline(...), ...}` with one
`Pipeline([("scaler", StandardScaler()), ("clf", <model>)])` each for:

- `LogisticRegression` (`max_iter=5000`)
- `KNeighborsClassifier`
- `SVC` (the RBF kernel is the default; `probability=True` so that ROC curves work later)
- `DecisionTreeClassifier` (`random_state=42`)
- `RandomForestClassifier` (`random_state=42`)
- `GradientBoostingClassifier` (`random_state=42`)

Script 2.4: trees/ensembles do not need scaling, but the pipeline does no harm and keeps
the code uniform (one loop for all models).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# TODO: models = {"Logistic regression": Pipeline([...]), ...} for all 6 models

print(list(models.keys()))
print(len(models) == 6)

## 4. A fair comparison by cross-validation

**Task:** for every model compute `cross_val_score` with `StratifiedKFold(n_splits=5,
shuffle=True, random_state=42)` and `scoring="roc_auc"` on the **training data**.
Collect the 5 scores per model (e.g. in `results = {"name": np.array([...]), ...}`)
and present them as a box plot.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TODO: results = {}
# TODO: for every (name, pipeline) in models.items(): compute cross_val_score(...) and store it

for name, scores in results.items():
    print(f"{name:25s}  ROC-AUC = {scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(results.values(), tick_labels=results.keys())
plt.xticks(rotation=30, ha="right")
plt.ylabel("ROC-AUC (5-fold CV)")
plt.title("The model race: CV ROC-AUC per model family")
plt.tight_layout()
plt.show()

**Task:** note briefly (2-3 sentences) in your own words: which 2 models go into
tuning, and why is the mean alone not enough — what do you watch for in the spread
(the width of the box)?

*(Your note here ...)*

## 5. Hyperparameter tuning of the top 2

**Task:** choose your two best models from step 4. Define a parameter grid for each
(script 2.5) and tune with `GridSearchCV` (`cv=cv`, `scoring="roc_auc"`) on the
**training data**. Examples of sensible grids:

- Random forest: `clf__n_estimators`, `clf__max_depth`
- Gradient boosting: `clf__n_estimators`, `clf__learning_rate`, `clf__max_depth`
- SVM: `clf__C`, `clf__gamma`
- Logistic regression: `clf__C`

(Parameter names in the grid need the pipeline prefix `clf__`, because the classifier
sits in the pipeline under the name `"clf"`.)

In [ ]:
from sklearn.model_selection import GridSearchCV

# TODO: grid_1 = {...}  parameter grid for your 1st top model
# TODO: search_1 = GridSearchCV(models["..."], grid_1, cv=cv, scoring="roc_auc", n_jobs=-1)
# TODO: search_1.fit(X_train, y_train)

print(f"Best CV ROC-AUC (model 1): {search_1.best_score_:.4f}")
print(f"Best parameters: {search_1.best_params_}")

In [ ]:
# TODO: the same for your 2nd top model (grid_2, search_2)

print(f"Best CV ROC-AUC (model 2): {search_2.best_score_:.4f}")
print(f"Best parameters: {search_2.best_params_}")

## 6. The one-off test evaluation

Only now does the test set come into play. **Task:** using `best_score_`, choose the
overall better of the two tuned models (`best_model = search_1.best_estimator_` or
`search_2.best_estimator_` — `GridSearchCV` has already refitted it on all of train).
Evaluate once on `X_test`/`y_test`: confusion matrix, classification report, ROC curve + AUC.

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay, roc_auc_score)

# TODO: best_model = ...
# TODO: y_pred = best_model.predict(X_test)
# TODO: y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["malignant", "benign"], ax=axes[0])
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.tight_layout()
plt.show()

# Mini check
print(roc_auc_score(y_test, y_proba) > 0.97)

## 7. Interpretation: permutation importance

**Task:** compute `permutation_importance` (from `sklearn.inspection`) for your final
model **on the test set** (script 3.1: that is the more honest place for it than the
training data). Plot the top 10 features as a horizontal bar chart.

In [ ]:
from sklearn.inspection import permutation_importance

# TODO: importance = permutation_importance(best_model, X_test, y_test,
#                                           n_repeats=30, random_state=42, scoring="roc_auc")

order = np.argsort(importance.importances_mean)[-10:]
plt.figure(figsize=(7, 5))
plt.barh(X.columns[order], importance.importances_mean[order],
         xerr=importance.importances_std[order])
plt.xlabel("drop in ROC-AUC when shuffled")
plt.title("Permutation importance (top 10, on test data)")
plt.tight_layout()
plt.show()

**Task:** do the top features seem medically plausible (e.g. `worst radius`,
`worst concave points`, `worst perimeter` — larger/more irregular cell nuclei point to
malignancy)? A short note:

*(Your note here ...)*

## 8. Learning curve: a bias or a variance problem?

**Task:** plot the learning curve (`learning_curve`, script 2.5) for your final model:
training and CV score against the amount of training data (`train_sizes=np.linspace(0.1, 1.0, 8)`,
`scoring="roc_auc"`, `cv=cv`, on `X_train`/`y_train`). Diagnose: is more data worth it?

In [ ]:
from sklearn.model_selection import learning_curve

# TODO: sizes, train_scores, val_scores = learning_curve(best_model, X_train, y_train,
#            train_sizes=np.linspace(0.1, 1.0, 8), cv=cv, scoring="roc_auc", n_jobs=-1)

plt.figure(figsize=(7, 4.5))
plt.plot(sizes, train_scores.mean(axis=1), marker="o", label="training")
plt.plot(sizes, val_scores.mean(axis=1), marker="s", label="validation (CV)")
plt.xlabel("training size"); plt.ylabel("ROC-AUC"); plt.legend()
plt.title("Learning curve of the final model")
plt.tight_layout()
plt.show()

**Task:** a bias or a variance problem (or neither)? Would more data help?

*(Your note here ...)*

## Done — what you can do now

- pack several model families consistently into pipelines
- compare models fairly by stratified cross-validation (read the mean **and** the spread)
- tune hyperparameters systematically with GridSearchCV without touching the test set
- test honestly **once** at the end and place the result in context with metrics, curves
  and permutation importance — including a learning curve diagnosis

**Reflection questions (answer each in 1-2 sentences):**

1. Why would it have been self-deception to run `GridSearchCV` directly with
   `scoring="accuracy"` on this data set without thinking about class balance?
2. Suppose `SVC` had had the best CV ROC-AUC in step 4 but the largest spread across the
   5 folds. Would you still choose it? What does that depend on?
3. What would change in step 6 if false positives were 10 times more expensive for this
   application than missed cases (false negatives)?

**Bonus tasks:**
1. Replace `GridSearchCV` with `RandomizedSearchCV` on a larger parameter space
   (e.g. distributions instead of lists) — compare run time and result.
2. Try `CalibratedClassifierCV` (script 3.2) on your final model and compare the
   reliability diagram before and after.